# Xarray 详细教案

---

## 目录

1. 课程概述
2. 环境搭建与安装
3. Xarray 核心概念
4. 第一课：DataArray 基础
5. 第二课：Dataset 数据结构
6. 第三课：索引与选择
7. 第四课：数据操作与计算
8. 第五课：数据聚合与统计
9. 第六课：可视化
10. 第七课：高级索引与重采样
11. 第八课：滚动窗口与加权计算
12. 第九课：文件读写
13. 第十课：与其他库的集成
14. 综合实战项目
15. 常见问题与最佳实践
16. 参考资源

---

## 1. 课程概述

### 1.1 什么是 Xarray？

Xarray 是一个强大的 Python 库，专为处理**多维标记数据集**而设计。它通过提供**命名维度**、**坐标标签**和**属性元数据**，扩展了 NumPy 的功能，使处理气候科学、海洋学和遥感等领域的复杂数据集变得更加直观和高效。

### 1.2 为什么选择 Xarray？

| 特性 | Xarray | NumPy |
|------|--------|-------|
| 标记维度 | 支持命名维度 | 仅支持整数索引 |
| 坐标系统 | 内置坐标标签 | 需要手动管理 |
| 元数据 | 属性存储 | 无内置支持 |
| 广播 | 自动对齐 | 需要手动处理 |
| 缺失值 | 原生支持 | 需要手动处理 |
| 分块计算 | Dask 集成 | 无内置支持 |

### 1.3 学习目标

完成本教案后，学员将能够：

- 理解 Xarray 的核心数据结构（DataArray 和 Dataset）
- 熟练使用标记索引选择和操作多维数据
- 执行数据聚合、统计和转换操作
- 可视化地理空间和时间序列数据
- 读写 netCDF、Zarr 等科学数据格式
- 将 Xarray 与 NumPy、Pandas、Matplotlib 等库结合使用

### 1.4 前置知识要求

- Python 基础语法
- NumPy 数组操作基础
- Pandas 数据结构基础
- 基本的地理信息系统（GIS）概念

---

## 2. 环境搭建与安装

### 2.1 使用 Conda 安装（推荐）

```bash
conda create -n xarray_env python=3.10
conda activate xarray_env
conda install -c conda-forge xarray numpy pandas matplotlib netcdf4 pooch
```

### 2.2 使用 pip 安装

```bash
python -m venv xarray_env
source xarray_env/bin/activate
pip install xarray numpy pandas matplotlib netcdf4 pooch
```

### 2.3 验证安装

In [1]:
import xarray as xr
import numpy as np
import pandas as pd

print(f"Xarray 版本: {xr.__version__}")
print(f"NumPy 版本: {np.__version__}")
print(f"Pandas 版本: {pd.__version__}")

---

## 3. Xarray 核心概念

### 3.1 数据结构对比

| 数据结构 | 描述 | 类比 |
|---------|------|------|
| **DataArray** | 标记的多维数组 | NumPy 数组 + 维度名称 + 坐标 + 属性 |
| **Dataset** | DataArray 的集合 | Pandas DataFrame 的多维版本 |

### 3.2 DataArray 结构

DataArray 由以下部分组成：
- **Values (值)**: 存储在 NumPy 数组中的实际数据
- **Dimensions (维度)**: 命名轴（例如 time, lat, lon）
- **Coordinates (坐标)**: 每个维度的标签
- **Attributes (属性)**: 元数据（单位、描述信息）
- **Name (名称)**: 数组的可选名称

### 3.3 Dataset 结构

Dataset 是一个类字典的 DataArray 集合，共享相同的维度和坐标。

---

## 4. 第一课：DataArray 基础

### 4.1 创建 DataArray

In [2]:
import xarray as xr
import numpy as np
import pandas as pd

# 创建数据：3 个时间步, 4 个纬度, 5 个经度的温度数据
data = np.random.rand(3, 4, 5) * 10 + 290

# 创建坐标
times = pd.date_range('2023-01-01', periods=3)
lats = np.arange(40, 44)
lons = np.arange(-105, -100)

# 创建 DataArray
da = xr.DataArray(
    data=data,
    dims=['time', 'lat', 'lon'],
    coords={'time': times, 'lat': lats, 'lon': lons},
    name='temperature',
    attrs={'units': 'K', 'description': 'Surface air temperature', 'source': 'Synthetic data'}
)

print(da)

### 4.2 访问 DataArray 组件

In [3]:
# 访问值 (返回 NumPy 数组)
print("值形状:", da.values.shape)
print("第一个元素:", da.values[0, 0, 0])

# 访问维度
print("维度名:", da.dims)
print("维度大小:", dict(da.sizes))

# 访问坐标
print("纬度坐标:", da['lat'].values)
print("经度坐标:", da['lon'].values)

# 访问属性
print("属性:", da.attrs)
print("单位:", da.attrs['units'])

# 访问名称
print("名称:", da.name)

### 4.3 坐标操作

In [4]:
# 添加新坐标
da_with_season = da.assign_coords(season=['冬季', '冬季', '冬季'])
print("添加季节坐标后:", list(da_with_season.coords))

# 修改属性
da.attrs['units'] = 'degC'
da.attrs['updated_by'] = 'Xarray教程'
print("修改后的属性:", da.attrs)

# 重命名
da_renamed = da.rename('temp')
print("重命名后:", da_renamed.name)

### 练习 4.1 - 创建降水 DataArray

In [5]:
# 练习：创建一个 7 天, 5x6 网格的降水数据 (0-15 mm/day)
precip_data = np.random.uniform(0, 15, (7, 5, 6))
times_p = pd.date_range('2023-06-01', periods=7)
lats_p = np.arange(30, 35)
lons_p = np.arange(110, 116)

precip = xr.DataArray(
    data=precip_data,
    dims=['time', 'lat', 'lon'],
    coords={'time': times_p, 'lat': lats_p, 'lon': lons_p},
    name='precipitation',
    attrs={'units': 'mm/day', 'description': 'Daily precipitation'}
)

print(f"名称: {precip.name}")
print(f"维度: {precip.dims}")
print(f"形状: {precip.shape}")
print(f"单位: {precip.attrs['units']}")
print(f"范围: {precip.values.min():.2f} ~ {precip.values.max():.2f} mm/day")
print(f"均值: {precip.values.mean():.2f} mm/day")

---

## 5. 第二课：Dataset 数据结构

### 5.1 创建 Dataset

In [6]:
times = pd.date_range('2023-01-01', periods=3)
lats = np.arange(40, 44)
lons = np.arange(-105, -100)

ds = xr.Dataset({
    'temperature': xr.DataArray(
        np.random.rand(3, 4, 5) * 10 + 290,
        dims=['time', 'lat', 'lon'],
        coords={'time': times, 'lat': lats, 'lon': lons},
        attrs={'units': 'K', 'description': '气温'}
    ),
    'humidity': xr.DataArray(
        np.random.rand(3, 4, 5) * 30 + 50,
        dims=['time', 'lat', 'lon'],
        coords={'time': times, 'lat': lats, 'lon': lons},
        attrs={'units': '%', 'description': '相对湿度'}
    ),
    'pressure': xr.DataArray(
        np.random.rand(3, 4, 5) * 5 + 1010,
        dims=['time', 'lat', 'lon'],
        coords={'time': times, 'lat': lats, 'lon': lons},
        attrs={'units': 'hPa', 'description': '海平面气压'}
    )
}, attrs={'title': '气象数据集', 'creator': 'Xarray教程', 'creation_date': '2023-01-15'})

print(ds)

### 5.2 Dataset 基本操作

In [7]:
print("变量列表:", list(ds.data_vars))
print("变量数:", len(ds.data_vars))

# 访问单个变量 (返回 DataArray)
temp = ds['temperature']
print(f"温度变量形状: {temp.shape}, 单位: {temp.attrs.get('units')}")

# 获取维度信息
print("维度大小:", dict(ds.dims))

# 获取全局属性
print("全局属性:", ds.attrs)

### 5.3 添加和删除变量

In [8]:
# 添加新变量 (风速)
ds['wind_speed'] = xr.DataArray(
    np.random.rand(3, 4, 5) * 10 + 3,
    dims=['time', 'lat', 'lon'],
    coords={'time': times, 'lat': lats, 'lon': lons},
    attrs={'units': 'm/s', 'description': '风速'}
)
print("添加风速后:", list(ds.data_vars))

# 删除变量
ds_no_wind = ds.drop_vars('wind_speed')
print("删除风速后:", list(ds_no_wind.data_vars))

# 选择部分变量
ds_subset = ds[['temperature', 'humidity']]
print("选择子集:", list(ds_subset.data_vars))

---

## 6. 第三课：索引与选择

### 6.1 标签索引 (.sel)

In [9]:
# 加载示例数据集（气温数据）
ds = xr.tutorial.open_dataset('air_temperature', engine='netcdf4')

print("原始数据大小:", ds['air'].shape)

# 选择特定时间点
temp_slice = ds['air'].sel(time='2013-01-01T12:00:00')
print(f"选择 2013-01-01 12:00 温度图，形状: {temp_slice.shape}")

# 选择特定位置 (返回时间序列)
ts = ds['air'].sel(lat=40, lon=260)
print(f"选择 (40N, 260E) 位置温度序列，形状: {ts.shape}")

# 选择时间范围
ds_jan = ds.sel(time=slice('2013-01-01', '2013-01-07'))
print(f"选择 1 月第一周: {len(ds_jan['time'])} 个时间步")

# 选择空间范围
ds_region = ds.sel(lat=slice(40, 60), lon=slice(250, 280))
print(f"选择北半球区域: {ds_region['air'].shape}")

### 6.2 位置索引 (.isel)

In [10]:
# 选择第一个时间步
first_step = ds['air'].isel(time=0)
print(f"第一个时间步形状: {first_step.shape}")

# 选择前 5 个时间步
first_5 = ds.isel(time=slice(0, 5))
print(f"前 5 个时间步: {len(first_5['time'])}")

# 选择特定网格点
subset = ds.isel(time=0, lat=slice(0, 5), lon=slice(0, 5))
print(f"选择第 1 时间步 + 前 5x5 网格: {subset['air'].shape}")

# 负索引：最后一个时间步
last_step = ds['air'].isel(time=-1)
print(f"最后一个时间步: {last_step['time'].values}")

### 6.3 高级索引 (最近邻 / 条件选择)

In [11]:
# 最近邻插值
nearest = ds['air'].sel(lat=40.1, lon=260.5, method='nearest')
print(f"请求 lat=40.1, lon=260.5")
print(f"实际选择 lat={nearest['lat'].values}, lon={nearest['lon'].values}")

# 条件选择 (where) - 不满足条件的位置设为 NaN
warm_air = ds['air'].where(ds['air'] > 300)
valid_count = np.count_nonzero(~np.isnan(warm_air.values))
print(f"温度 > 300K 的有效点数: {valid_count} / {warm_air.size}")

# 使用 .loc 进行标签选择 (类似 Pandas)
loc_selection = ds['air'].loc['2013-06-01':'2013-06-03', :, :]
print(f"使用 .loc 选择 6 月 1-3 日: {loc_selection.shape}")

---

## 7. 第四课：数据操作与计算

### 7.1 算术运算

In [12]:
# 单位转换: K -> C
temp_c = ds['air'] - 273.15
temp_c.attrs = {'units': 'degC', 'description': '气温(摄氏度)'}
print(f"原始范围: {ds['air'].values.min():.2f} ~ {ds['air'].values.max():.2f} K")
print(f"转换后范围: {temp_c.values.min():.2f} ~ {temp_c.values.max():.2f} C")

# 计算温度异常值 (距平)
mean_temp = ds['air'].mean(dim='time')
anomaly = ds['air'] - mean_temp
print(f"异常值形状: {anomaly.shape}")
print(f"异常值范围: {anomaly.values.min():.2f} ~ {anomaly.values.max():.2f} K")
print(f"异常值均值: {anomaly.values.mean():.4f} K (理论上应接近 0)")

# 纬度方向差分
temp_grad = ds['air'].diff(dim='lat')
print(f"纬度梯度形状: {temp_grad.shape} (纬度维度减少 1)")

### 7.2 广播 (Broadcasting)

In [13]:
# 创建 1D 纬度权重 (高纬度权重更小)
lat_weights = xr.DataArray(
    np.cos(np.deg2rad(ds['lat'])),
    dims=['lat'],
    coords={'lat': ds['lat']},
    name='lat_weight'
)
print(f"纬度权重形状: {lat_weights.shape} (1D)")
print(f"纬度权重范围: {lat_weights.values.min():.4f} ~ {lat_weights.values.max():.4f}")

# 1D 权重与 3D 温度数据相乘 (自动广播)
weighted_temp = ds['air'] * lat_weights
print(f"广播后形状: {weighted_temp.shape} (3D)")

# 使用 apply_ufunc 自定义函数
def celsius_to_fahrenheit(x):
    return (x - 273.15) * 9/5 + 32

temp_f = xr.apply_ufunc(celsius_to_fahrenheit, ds['air'])
temp_f.attrs = {'units': 'degF', 'description': '气温(华氏度)'}
print(f"K->F 转换后范围: {temp_f.values.min():.1f} ~ {temp_f.values.max():.1f} F")

---

## 8. 第五课：数据聚合与统计

### 8.1 基础统计

In [14]:
# 时间维度求平均 (得到空间图)
mean_time = ds['air'].mean(dim='time')
print(f"时间平均后形状: {mean_time.shape} (lat, lon)")

# 空间维度求平均 (得到时间序列)
mean_space = ds['air'].mean(dim=['lat', 'lon'])
print(f"空间平均后形状: {mean_space.shape} (time,)")
print(f"前 3 时间步全球平均: {mean_space.values[:3]}")

# 多种统计量
print(f"温度最大值: {ds['air'].values.max():.2f} K")
print(f"温度最小值: {ds['air'].values.min():.2f} K")
print(f"温度标准差: {ds['air'].values.std():.2f} K")
print(f"温度中位数: {np.median(ds['air'].values):.2f} K")

# 使用 .agg 批量计算
stats = ds['air'].agg(['mean', 'std', 'min', 'max'])
print(f"agg 统计变量: {list(stats.data_vars)}")

### 8.2 GroupBy 分组运算

In [15]:
# 按季节分组
seasonal_mean = ds.groupby('time.season').mean()
print(f"季节: {list(seasonal_mean['season'].values)}")
print(f"季节平均形状: {seasonal_mean['air'].shape}")
for season in seasonal_mean['season'].values:
    sd = seasonal_mean['air'].sel(season=season)
    print(f"  {season}: {sd.values.mean():.2f} K 平均温度")

# 按月份分组
monthly_mean = ds.groupby('time.month').mean()
print(f"月份: {list(monthly_mean['month'].values)}")
print(f"月份平均形状: {monthly_mean['air'].shape}")
for m in monthly_mean['month'].values[:3]:
    md = monthly_mean['air'].sel(month=m)
    print(f"  第 {m} 月: {md.values.mean():.2f} K")

# 按年内天数分组
dayofyear_mean = ds.groupby('time.dayofyear').mean()
print(f"按日分组形状: {dayofyear_mean['air'].shape}")

---

## 9. 第六课：可视化

### 9.1 基础绘图（空间图）

In [16]:
import matplotlib.pyplot as plt

# 时间平均温度图
mean_temp = ds['air'].mean(dim='time')
fig, ax = plt.subplots(figsize=(10, 6))
mean_temp.plot(ax=ax, cmap='viridis')
ax.set_title('时间平均温度 (Mean Air Temperature)')
ax.set_xlabel('经度')
ax.set_ylabel('纬度')
plt.tight_layout()
plt.show()

# 特定时间点温度图
fig, ax = plt.subplots(figsize=(10, 6))
ds['air'].isel(time=0).plot(ax=ax, cmap='RdBu_r', robust=True)
ax.set_title(f'温度图 - {ds["time"].values[0]}')
plt.tight_layout()
plt.show()

### 9.2 时间序列绘图

In [17]:
# 单个位置的时间序列
ts = ds['air'].sel(lat=40, lon=260)

fig, ax = plt.subplots(figsize=(14, 5))
ts.plot(ax=ax, linewidth=1, color='steelblue')
ax.set_title('温度时间序列 - (40N, 260E)')
ax.set_ylabel('温度 (K)')
ax.set_xlabel('时间')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 多个位置的时间序列
fig, ax = plt.subplots(figsize=(14, 5))
ds['air'].sel(lat=[60, 40, 20], lon=260).plot.line(x='time', ax=ax)
ax.set_title('不同纬度的温度时间序列')
ax.set_ylabel('温度 (K)')
ax.legend(['60N', '40N', '20N'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 9.3 分面图 (Faceted Plots)

In [18]:
# 季节平均的分面图
seasonal = ds.groupby('time.season').mean()

# 按季节分面
seasonal['air'].plot(col='season', col_wrap=2, cmap='viridis', figsize=(12, 8))
plt.suptitle('季节平均温度 (Seasonal Mean Temperature)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# 直方图
fig, ax = plt.subplots(figsize=(10, 5))
ds['air'].isel(time=slice(0, 10)).plot.hist(bins=30, ax=ax, alpha=0.7, color='steelblue', edgecolor='white')
ax.set_title('温度分布直方图 (前 10 个时间步)')
ax.set_xlabel('温度 (K)')
ax.set_ylabel('频数')
plt.tight_layout()
plt.show()

---

## 10. 第七课：高级索引与重采样

### 10.1 时间重采样

In [19]:
# 每日 -> 月平均
monthly = ds.resample(time='1MS').mean()
print(f"原始时间步: {len(ds['time'])}")
print(f"月平均时间步: {len(monthly['time'])}")
print(f"月份: {[str(t)[:7] for t in monthly['time'].values]}")

# 月统计量 (同时计算 mean, std, min, max)
monthly_stats = ds['air'].resample(time='1MS').agg(['mean', 'std', 'min', 'max'])
print(f"月度统计 - 统计变量: {list(monthly_stats.data_vars)}")

# 年最大值
annual_max = ds['air'].resample(time='1AS').max()
print(f"年数: {len(annual_max['time'])}")
for i, t in enumerate(annual_max['time'].values):
    print(f"  {str(t)[:4]}: 最高温度 = {annual_max.values[i].max():.2f} K")

### 10.2 滚动窗口（平滑）

In [20]:
import matplotlib.pyplot as plt

# 7 天滚动平均
rolling_mean = ds['air'].rolling(time=7, center=True).mean()
print(f"7 天滚动平均形状: {rolling_mean.shape}")
print(f"边缘 NaN 数: {np.isnan(rolling_mean.values[:, 0, 0]).sum()}")

# 30 天滚动标准差
rolling_std = ds['air'].rolling(time=30, center=True).std()
print(f"30 天滚动标准差: 平均 {np.nanmean(rolling_std.values):.2f} K")

# 可视化对比
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
ts_raw = ds['air'].sel(lat=40, lon=260)
ts_smooth = rolling_mean.sel(lat=40, lon=260)

ts_raw.plot(ax=axes[0], linewidth=0.8, color='steelblue', label='原始数据')
axes[0].set_title('原始温度时间序列')
axes[0].set_ylabel('温度 (K)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

ts_smooth.plot(ax=axes[1], linewidth=1.2, color='darkred', label='7 天滚动平均')
axes[1].set_title('7 天滚动平均温度')
axes[1].set_ylabel('温度 (K)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 11. 第八课：滚动窗口与加权计算

### 11.1 加权运算

In [21]:
import matplotlib.pyplot as plt

# 纬度权重 (地球是球面, 高纬度网格面积较小)
weights = np.cos(np.deg2rad(ds['lat']))
weights.name = 'weights'
print(f"纬度范围: {ds['lat'].values.min():.1f} ~ {ds['lat'].values.max():.1f}")
print(f"权重范围: {weights.values.min():.4f} ~ {weights.values.max():.4f}")

# 加权的空间平均
weighted_mean = ds.weighted(weights).mean(dim=['lat', 'lon'])
unweighted_mean = ds.mean(dim=['lat', 'lon'])
print(f"加权平均温度: {weighted_mean['air'].values.mean():.2f} K")
print(f"不加权平均温度: {unweighted_mean['air'].values.mean():.2f} K")
print(f"差异: {weighted_mean['air'].values.mean() - unweighted_mean['air'].values.mean():.4f} K")

# 可视化对比
fig, ax = plt.subplots(figsize=(14, 5))
weighted_mean['air'].plot(ax=ax, label='纬度加权平均', linewidth=1.5, color='steelblue')
unweighted_mean['air'].plot(ax=ax, label='简单平均', linewidth=1.0, color='orange', linestyle='--')
ax.set_title('全球平均温度: 加权 vs 不加权')
ax.set_ylabel('温度 (K)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 12. 第九课：文件读写

### 12.1 写入 netCDF

In [22]:
# 选择子集以便快速写入
ds_subset = ds.isel(time=slice(0, 10))
ds_subset.attrs['title'] = '气温数据集示例'
ds_subset.attrs['creator'] = 'Xarray教程'

# 写入 netCDF
out_path = 'air_temperature_subset.nc'
ds_subset.to_netcdf(out_path)
print(f"已保存到: {out_path}")
print(f"  变量: {list(ds_subset.data_vars)}")
print(f"  形状: {ds_subset['air'].shape}")

# 保存多个分块文件
for i in range(3):
    ds_chunk = ds.isel(time=slice(i*5, (i+1)*5))
    chunk_path = f'air_temperature_chunk_{i+1:02d}.nc'
    ds_chunk.to_netcdf(chunk_path)
    print(f"  已保存: {chunk_path} (时间步: {len(ds_chunk['time'])})")

### 12.2 读取 netCDF

In [23]:
# 读取 netCDF
loaded_ds = xr.open_dataset('air_temperature_subset.nc')
print(f"从文件读取:")
print(f"  变量: {list(loaded_ds.data_vars)}")
print(f"  形状: {loaded_ds['air'].shape}")
print(f"  全局属性: {loaded_ds.attrs}")

# 使用 with 语句读取
with xr.open_dataset('air_temperature_subset.nc') as f:
    temp_data = f['air'].values
print(f"使用 with 读取: 温度数据形状 {temp_data.shape}")

# 读取多个文件并合并 (open_mfdataset)
paths = ['air_temperature_chunk_01.nc', 'air_temperature_chunk_02.nc', 'air_temperature_chunk_03.nc']
ds_merged = xr.open_mfdataset(paths, combine='by_coords')
print(f"多文件合并后形状: {ds_merged['air'].shape}")
print(f"合并后时间步数: {len(ds_merged['time'])}")

### 12.3 Zarr 格式 (云优化存储)

In [24]:
# Zarr 是面向云的分块存储格式, 适合大型数据集
try:
    import zarr
    zarr_path = 'air_temperature.zarr'
    ds_subset.to_zarr(zarr_path, mode='w', consolidated=True)
    print(f"已保存到 Zarr: {zarr_path}")
    ds_zarr = xr.open_zarr(zarr_path)
    print(f"从 Zarr 读取, 形状: {ds_zarr['air'].shape}")
except ImportError:
    print("需要安装 zarr: pip install zarr")
    print("跳过 Zarr 示例...")

---

## 13. 第十课：与其他库的集成

### 13.1 转换为 NumPy

In [25]:
# DataArray -> NumPy 数组 (失去坐标信息)
numpy_array = ds['air'].values
print(f"NumPy 数组形状: {numpy_array.shape}")
print(f"NumPy 数组 dtype: {numpy_array.dtype}")
print(f"前 3x3 元素:")
print(numpy_array[0, :3, :3])

# 使用 NumPy 函数处理 Xarray 数据
temp_celsius_np = np.round(ds['air'].values - 273.15, 1)
print(f"转换为摄氏度 (保留 1 位小数): {temp_celsius_np.min():.1f} ~ {temp_celsius_np.max():.1f} C")

# 保存为二进制
np.save('temp_array.npy', ds['air'].isel(time=0).values)
print("已保存 NumPy 数组到 temp_array.npy")

### 13.2 转换为 Pandas

In [26]:
# 单个位置时间序列 -> Pandas Series
ts = ds['air'].sel(lat=40, lon=260)
series = ts.to_pandas()
print(f"Pandas Series 长度: {len(series)}")
print(f"前 5 个值:")
print(series.head())

# DataArray -> Pandas DataFrame
df = ds['air'].isel(time=slice(0, 3)).to_dataframe()
print(f"\nPandas DataFrame 形状: {df.shape}")
print(f"DataFrame 列名: {list(df.columns)}")
print(df.head())

# 使用 Pandas 进行时间序列分析 (例如 30 天移动平均)
rolling30 = series.rolling(window=30).mean()
print(f"\nPandas 30 天移动平均 - 前 5 个值:")
print(rolling30.head())

---

## 14. 综合实战项目

### 项目：气温数据分析完整流程

目标：加载数据 -> 季节分析 -> 查找极值 -> 可视化 -> 保存结果

In [4]:
import matplotlib.pyplot as plt
import xarray as xr
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12

# 步骤 1: 加载数据集
ds = xr.tutorial.open_dataset('air_temperature', engine='netcdf4')
print("=== 步骤 1: 加载数据 ===")
print(f"数据集形状: {ds['air'].shape}")
print(f"时间范围: {ds['time'].values[0]} ~ {ds['time'].values[-1]}")

# 步骤 2: 季节平均
seasonal = ds.groupby('time.season').mean()
print("\n=== 步骤 2: 季节平均 ===")
print(f"季节: {list(seasonal['season'].values)}")

# 步骤 3: 查找最高/最低温度位置
mean_temp = ds['air'].mean(dim='time')
max_temp = mean_temp.max().values
min_temp = mean_temp.min().values
max_loc = np.unravel_index(np.argmax(mean_temp.values), mean_temp.shape)
min_loc = np.unravel_index(np.argmin(mean_temp.values), mean_temp.shape)
print(f"\n=== 步骤 3: 查找极值 ===")
print(f"最高平均温度: {max_temp:.2f} K 在 lat={mean_temp['lat'].values[max_loc[0]]:.1f}, lon={mean_temp['lon'].values[max_loc[1]]:.1f}")
print(f"最低平均温度: {min_temp:.2f} K 在 lat={mean_temp['lat'].values[min_loc[0]]:.1f}, lon={mean_temp['lon'].values[min_loc[1]]:.1f}")

# 步骤 4: 可视化
print("\n=== 步骤 4: 可视化 ===")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
seasons = ['DJF', 'MAM', 'JJA', 'SON']
season_names = {'DJF': '冬季 (12-2月)', 'MAM': '春季 (3-5月)', 'JJA': '夏季 (6-8月)', 'SON': '秋季 (9-11月)'}

for i, season in enumerate(seasons):
    row, col = i // 2, i % 2
    data = seasonal['air'].sel(season=season)
    im = axes[row, col].pcolormesh(data['lon'], data['lat'], data.values, cmap='viridis')
    axes[row, col].set_title(f'{season} ({season_names[season]})')
    axes[row, col].set_xlabel('经度')
    axes[row, col].set_ylabel('纬度')
    plt.colorbar(im, ax=axes[row, col], label='温度 (K)')

plt.suptitle('季节平均温度分布', fontsize=16)
plt.tight_layout()
plt.show()

# 步骤 5: 保存结果
seasonal.attrs['title'] = '季节平均气温数据集'
seasonal.attrs['creator'] = 'Xarray教程-综合项目'
seasonal.to_netcdf('seasonal_temperature.nc')
print("\n=== 步骤 5: 保存结果 ===")
print("已保存到 seasonal_temperature.nc")
print("\n=== 项目完成! ===")

ReadTimeout: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Read timed out. (read timeout=30)

---

## 15. 常见问题与最佳实践

### 15.1 最佳实践

1. **使用标记索引**: 优先使用 .sel() 而不是 .isel() 以提高可读性
2. **保留属性**: 完善并保留变量属性 (units, description 等)
3. **懒惰求值**: Xarray 默认懒惰求值, 使用 .compute() 或 .values 触发计算
4. **分块计算**: 大型数据集使用 Dask 进行分块并行计算
5. **纬度加权**: 计算全球平均时使用 cos(纬度) 权重
6. **完善元数据**: 在保存 netCDF 前填写全局属性 (title, creator, history 等)

### 15.2 常见问题

- **维度不匹配**: 组合数据集时确保维度对齐
- **内存不足**: 使用 chunk() 或分块读取, 或先裁剪子区域
- **时区问题**: 尽量使用 UTC 时间, 避免时区转换
- **坐标非单调**: 使用 sortby() 确保坐标有序
- **属性丢失**: 算术运算后属性会丢失, 需要手动重新设置或使用 xr.set_options(keep_attrs=True)

---

## 16. 参考资源

- Xarray 官方文档: https://docs.xarray.dev
- Xarray 教程: https://tutorial.xarray.dev
- Pangeo 教程: https://pangeo.io
- Python 气象数据处理: https://unidata.github.io/python-training
- netCDF 格式规范: https://www.unidata.ucar.edu/software/netcdf

---

**恭喜！你已完成 Xarray 详细教程的学习。**

建议下一步：
- 尝试使用真实的 netCDF 数据 (ERA5, CMIP6, NOAA 等)
- 探索 Dask 并行计算处理 TB 级数据集
- 学习地理投影与 Cartopy 结合制图